# 02. Preprocessing (steps ①-⑤)

| | |
|---|---|
| Input | Frame-aligned top and front videos, YOLO weights, calibration NPZ |
| Output | `3D_raw_online.xlsx`, `analyzed_output.mp4` (optional), `verify_unification.png`, `config_used.yaml` |
| GPU | Required (Runtime → Change runtime type → T4 GPU) |

Steps run frame by frame in `main_code/preprocessing/run_preprocessing.py`:
① `step01_detect_object` → ② `step02_refract_rays` → ③ `step03_triangulate` → ④ `step04_gate_and_track` → ⑤ `step05_to_gripper_frame`

In [ ]:
#@title 0. Setup (clone repository, install packages, mount Drive)
REPO_URL = "https://github.com/dodokim777-boop/underwater-gripper-tracking.git"  #@param {type:"string"}

import os, sys, subprocess, warnings
REPO_DIR = '/content/' + REPO_URL.rstrip('/').split('/')[-1].removesuffix('.git')
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=False)
%pip install -q -r {REPO_DIR}/requirements.txt

from google.colab import drive
drive.mount('/content/drive')

sys.path.insert(0, REPO_DIR)
import pandas as pd
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
print('Ready:', REPO_DIR)

In [ ]:
#@title 1. Settings file
SETTINGS_PATH = "/content/drive/MyDrive/files/experiment_settings.yaml"  #@param {type:"string"}

import shutil
from main_code import settings

if not os.path.exists(SETTINGS_PATH):
    os.makedirs(os.path.dirname(SETTINGS_PATH), exist_ok=True)
    shutil.copy(os.path.join(REPO_DIR, 'experiment_settings.yaml'), SETTINGS_PATH)
    raise RuntimeError(f'Template copied to {SETTINGS_PATH}. Edit the required values and run this cell again.')

settings.configure(SETTINGS_PATH)
settings.print_summary('preprocessing')
settings.check('preprocessing')

In [ ]:
#@title 2. GPU check
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not available (very slow on CPU)')

## 3. Test run

Processes the first frames only to check paths, models, and calibration.
In `verify_unification.png`, the red dots should sit on the floor marker corners.

In [ ]:
#@title 3. Test run
TEST_FRAMES = 300  #@param {type:"integer"}
import cv2, matplotlib.pyplot as plt
from main_code.preprocessing.run_preprocessing import run_preprocessing

df_test = run_preprocessing(max_frames=TEST_FRAMES)
if os.path.exists(settings.VERIFY_IMAGE_PATH):
    plt.figure(figsize=(12, 7))
    plt.imshow(cv2.cvtColor(cv2.imread(settings.VERIFY_IMAGE_PATH), cv2.COLOR_BGR2RGB))
    plt.axis('off'); plt.show()

In [ ]:
#@title 4. Full run
from main_code.preprocessing.run_preprocessing import run_preprocessing
df_raw = run_preprocessing()

In [ ]:
#@title 5. Result check
import matplotlib.pyplot as plt
print(df_raw['Missing_Reason'].value_counts().to_string())
fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
for ax, c in zip(axes, ['Meas_X', 'Meas_Y', 'Meas_Z']):
    ax.plot(df_raw['Time_sec'], df_raw[c], '.', ms=2)
    ax.set_ylabel(c + ' [cm]'); ax.grid(alpha=.3)
axes[-1].set_xlabel('Time [s]'); plt.tight_layout(); plt.show()